In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','products','Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(catalog,data_source)


In [0]:
df = (spark.read.format('csv')
      .option('inferSchema','true')
      .option('header','true')
      .load('/Volumes/fmcg/bronze/source_fmcg/products/')
      .withColumn('read_timestamp',F.current_timestamp())
      .select("*","_metadata.file_name",'_metadata.file_size')
      
      )
      

In [0]:
df.display()

In [0]:
df.write.format('delta').option('delta.enableChangeDataFeed','true').mode('overwrite').saveAsTable('fmcg.bronze.products')

# **_Silver tranformation_**

In [0]:
df_bronze = spark.sql('select * from fmcg.bronze.products')
df_bronze.display(5)

# **Drop Duplicates**


In [0]:
print('Rows before drop duplicates',df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print('Rows after drop duplicates',df_silver.count())

Title case fix 
(energy bars --> Energy Bars, protien bars ---> Protien Bars)

In [0]:
df_silver.select("category").distinct().show()

In [0]:
df_silver = df_silver.withColumn('category',
                     F.when(F.col('category').isNull(),None)
                     .otherwise(F.initcap('category'))
                     
                     
                     )
df_silver.select('category').distinct().display()

# **Fix spelling Mistake for Protien**

In [0]:
df_silver = (
  df_silver.withColumn(
    'product_name',F.regexp_replace(F.col('product_name'),"(?i)Protien",'Protein')
  )
  .withColumn(
    'category',
    F.regexp_replace(F.col('category'),"(?i)Protien","Protein")
  )
)

In [0]:
df_silver = (
    df_silver.withColumn(
        'division',
        F.when(F.col('category') == "Energy Bars", "Nutrition Bars")
        .when(F.col('category') == "Protein Bars","Nutrition Bars")
        .when(F.col('category') == "Granola & Cereals", "Breakfast Foods")
        .when(F.col('category') == "Recovery Dairy","Dairy & Recovery")
        .when(F.col('category') == "Healthy Snacks", "Healthy Snacks")
        .when(F.col('category') == "Electrolyte Mix",'Hydration & Electrolytes')
        .otherwise('other')
    )
)

# **Variant Column**

In [0]:
df_silver = df_silver.withColumn(
  'variant',
  F.regexp_extract(F.col('product_name'),r'\((.*?)\)',1)
)

Invalid product_id are repalced with a fallaback value to avpid lossing fct records and ensure downstream joins remain consistent

In [0]:
df_silver = (
    df_silver.withColumn(
        'product_code',
        F.sha2(F.col('product_name').cast('string'),256)
    )
    .withColumn(
        "product_id",
        F.when(
            F.col('product_id').cast('string').rlike("^[0-9]+$"),
            F.col('product_id').cast('string')
        ).otherwise(F.lit(999999).cast('string'))

    )
)

In [0]:
df_silver = df_silver.select('product_code','division','category','product_name','variant','product_id','read_timestamp','file_name','file_size')

In [0]:
df_silver.display()

In [0]:
df_silver.write\
    .mode('delta')\
        .option('delta.enableChangeDataFeed',"true")\
            .option('mergeSchema','true')\
                .mode('overwrite')\
                    .saveAsTable('fmcg.silver.products')

# **Gold**

In [0]:
df_silver = spark.sql('select * from fmcg.silver.products')
df_gold = df_silver.select('product_code','product_id','division','category','product_name','variant')
df_gold.display()

In [0]:
parent_delta_table = DeltaTable.forName(spark,'fmcg.gold.dim_products')
df_child_products = spark.sql('select product_code,division,category,product,variant from fmcg.gold.dim_products')
df_child_products.display(5)



parent_delta_table.alias('target').merge(
    source = df_child_products.alias('source'),
    condition='target.product_code = source.product_code'
).whenMatchedUpdate(
    set = {
        'division': 'source.division',
        'category': 'source.category',
        'product': 'source.product',
        'variant': 'source.variant'
    }
).whenNotMatchedInsert(
    values = {
        'product_code': 'source.product_code',
        'division' : 'source.division',
        'category' : 'source.category',
        'variant' : 'source.variant'

    }
).execute()